In [5]:
# libararies

# Tensorflow 2
import tensorflow as tf
from tensorflow import keras
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import GaussianNoise, Dropout


from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten


import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline  

from PIL import Image
import glob

import os
import random
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import f1_score

from tensorflow.keras.models import Model

from tf_keras_vis.saliency import Saliency
from tf_keras_vis.utils import normalize

from tf_keras_vis.saliency import Saliency
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.scorecam import Scorecam
from tf_keras_vis.activation_maximization import ActivationMaximization
from tf_keras_vis.activation_maximization.callbacks import Progress
from tf_keras_vis.activation_maximization.input_modifiers import Jitter, Rotate2D
from tf_keras_vis.activation_maximization.regularizers import TotalVariation2D, Norm
from tf_keras_vis.utils.model_modifiers import ExtractIntermediateLayer, ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore


# original (wxh) is (800x600)
w = 224
h = 224

# fix random seed for reproducibility
SEED = 42

# Loading Segmented Leaves

In [7]:
import os, glob
import numpy as np
from PIL import Image

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def list_images(folder):
    return sorted(os.path.join(folder, f) for f in os.listdir(folder)
                  if os.path.splitext(f)[1].lower() in IMG_EXTS)

FOLDERS = {'../olive_leaf_dataset/segmented/train/Healthy': 0,
           '../olive_leaf_dataset/segmented/train/aculus_olearius': 1,
           '../olive_leaf_dataset/segmented/train/olive_peacock_spot': 2}

paths, X, Y = [], [], []
seg_info_rows = []
for folder, label in FOLDERS.items():
    for filename in list_images(folder):
        im = Image.open(filename).convert('RGB').resize((w, h), Image.LANCZOS)
        paths.append(filename)
        X.append(np.array(im))
        Y.append(label)


# Convert to NP array
X = np.array(X)


# reshape to be [samples][channels][width][height]
X = X.reshape(X.shape[0], w, h, 3).astype('float32')

# Normalize the data
X = X /255

Y = np.array(Y)

# randomize the data set - numpy arrays
randomize = np.arange(len(X))
np.random.shuffle(randomize)
X = X[randomize]
Y = Y[randomize]
# `paths` has to move with them. Without this line the exported CSV pairs each
# feature row with a different image's filename, which makes a prediction
# impossible to trace back to a leaf.
paths = [paths[i] for i in randomize]


# one hot encode outputs
Y_orig = Y
Y = to_categorical(Y)
num_classes = Y.shape[1]

print("X:", X.shape, "| Y:", Y.shape, "| per-class:", Y.sum(axis=0))

X: (3003, 224, 224, 3) | Y: (3003, 3) | per-class: [        899         761        1343]


# Preprocessing

## Leaf Feature Extraction

In this section, texture and color fraction of the leaves are extracted. GLCM is used to get the texture of the leaves.

In [8]:
import numpy as np
from skimage.color import rgb2hsv, rgb2gray
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.measure import label, regionprops
from skimage.filters import sobel
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

def leaf_features(img):
    """
    img: HxWx3 float in [0,1]  (i.e. your X[i] after /255)
    returns: dict of named features
    """
    img = np.asarray(img, dtype=float)
    if img.max() > 1.5:                      # tolerate uint8 input too
        img = img / 255.0
    R, G, B = img[..., 0], img[..., 1], img[..., 2]
    hsv = rgb2hsv(img)
    Hh, S, V = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]
    gray = rgb2gray(img)
    f = {}
 
    # ---- leaf mask: drop near-white / near-black background --------------
    leaf = (V > 0.15) & ~((S < 0.15) & (V > 0.85))
    if leaf.sum() < 100:                     # fallback if masking fails
        leaf = np.ones_like(V, dtype=bool)
    #f["leaf_area_frac"] = leaf.mean()
    n_leaf = leaf.sum()
 
    # ---- 1. tissue colour composition (all measured inside the leaf) -----
    def frac(mask):
        return (mask & leaf).sum() / n_leaf
 
    f["frac_green"]  = frac((Hh >= 70) & (Hh < 160) & (S > 0.25))
    f["frac_yellow"] = frac((Hh >= 40) & (Hh < 70) & (S > 0.25))   # chlorosis
    f["frac_brown"]  = frac((Hh >= 10) & (Hh < 40) & (S > 0.20) & (V < 0.6))
    f["frac_dark"]   = frac(V < 0.25)                              # necrosis
    f["frac_gray"]   = frac(S < 0.20)                              # silvering
 
    # ---- 2. colour statistics inside the leaf ---------------------------
    #f["mean_hue"] = Hh[leaf].mean()
    #f["std_hue"] = Hh[leaf].std()            # mottling -> high
    #f["mean_sat"] = S[leaf].mean()
    #f["std_sat"] = S[leaf].std()
    #f["mean_val"] = V[leaf].mean()
    #f["std_val"] = V[leaf].std()
    #f["excess_green"] = (2 * G - R - B)[leaf].mean()
    #f["green_minus_red"] = (G - R)[leaf].mean()
 
    # ---- 3. lesion morphology (the peacock-spot signature) --------------
    # dark or brown blobs sitting inside otherwise green tissue
    #lesion = leaf & ((V < 0.35) | ((Hh >= 10) & (Hh < 45) & (S > 0.25) & (V < 0.55)))
    #lab = label(lesion)
    #props = [p for p in regionprops(lab) if p.area >= 20]
    #f["lesion_area_frac"] = lesion.sum() / n_leaf
    #f["n_lesions"] = len(props)
    #if props:
    #    areas = np.array([p.area for p in props])
    #    f["lesion_mean_area"] = areas.mean() / n_leaf
    #    f["lesion_max_area"] = areas.max() / n_leaf
    #   f["lesion_area_std"] = areas.std() / n_leaf
    #    circ = [4 * np.pi * p.area / max(p.perimeter ** 2, 1e-6) for p in props]
    #    f["lesion_circularity"] = float(np.mean(circ))
    #    f["lesion_eccentricity"] = float(np.mean([p.eccentricity for p in props]))
    #else:
    #    for k in ["lesion_mean_area", "lesion_max_area", "lesion_area_std",
      #            "lesion_circularity", "lesion_eccentricity"]:
       #     f[k] = 0.0
 
    # ---- 4. texture: GLCM (roughness / mottling) ------------------------
    q = (gray * 63).astype(np.uint8)
    glcm = graycomatrix(q, distances=[1, 3], angles=[0, np.pi / 2],
                        levels=64, symmetric=True, normed=True)
    for prop in ["contrast", "homogeneity", "energy", "correlation"]:
        f[f"glcm_{prop}"] = float(graycoprops(glcm, prop).mean())
 
    # ---- 5. texture: LBP histogram (surface pattern) --------------------
    #lbp = local_binary_pattern((gray * 255).astype(np.uint8), P=8, R=1,
    #                           method="uniform")
    #hist, _ = np.histogram(lbp[leaf], bins=10, range=(0, 10), density=True)
    #for i, v in enumerate(hist):
    #    f[f"lbp_{i}"] = float(v)
 
    # ---- 6. edges (lesion borders, vein prominence) ---------------------
    #edges = sobel(gray)
    #f["edge_mean"] = edges[leaf].mean()
    #f["edge_frac_strong"] = (edges[leaf] > 0.08).mean()
 
    return f

In [9]:
def build_matrix(images):
    dicts = [leaf_features(im) for im in images]
    names = list(dicts[0].keys())
    return np.array([[d[k] for k in names] for d in dicts]), names

In [10]:
import pandas as pd

X_feat, feature_names = build_matrix(X)

CLASS_NAMES = {0: 'Healthy', 1: 'aculus_olearius', 2: 'olive_peacock_spot'}

df = pd.DataFrame(X_feat, columns=feature_names)
df.insert(0, 'file',  [os.path.basename(p) for p in paths])
df.insert(1, 'class', [CLASS_NAMES[v] for v in Y_orig])
df.insert(2, 'label', Y_orig)

print(df.shape)      # (2720, 40)
df.head()

(3003, 12)


,file,class,label,frac_green,frac_yellow,frac_brown,frac_dark,frac_gray,glcm_contrast,glcm_homogeneity,glcm_energy,glcm_correlation
0,A163_leaf00.png,olive_peacock_spot,2,0.184844,0.689449,0.000969,0.006625,0.065277,8.560721,0.921606,0.869085,0.951537
1,A299_leaf00.png,aculus_olearius,1,0.932814,0.016125,0.000000,0.074308,0.031443,11.091058,0.887516,0.842724,0.939271
2,A310_leaf00.png,Healthy,0,0.365580,0.449537,0.001543,0.011679,0.033715,9.542445,0.936641,0.902282,0.926680
3,A319_leaf00.png,Healthy,0,0.632177,0.164243,0.112416,0.265474,0.076249,9.406268,0.919346,0.882557,0.904804
4,A442_leaf00.png,Healthy,0,0.013461,0.000299,0.000000,0.092133,0.868681,4.822011,0.956873,0.928116,0.929371


In [12]:
df.to_csv('leaf_features.csv', index=False)

# Train model - Decision Tree

In [ ]:
# The segmenter emits one crop per detected leaf, so a single photograph yields
# `<stem>_leaf00`, `<stem>_leaf01`, ... 550 of the 3,003 crops (18.3%) come from
# the 267 photographs that produced more than one. A plain train_test_split
# scatters those siblings across both sides and near-identical crops of one leaf
# then sit in train and test at once, which inflates the score. Grouping on the
# source photograph prevents that, and StratifiedGroupKFold also restores the
# stratification the previous call left out.
import re
from sklearn.model_selection import StratifiedGroupKFold

groups = np.array([re.sub(r'_leaf\d+\.(png|jpg|jpeg)$', '', os.path.basename(p), flags=re.I)
                   for p in paths])
splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
train_idx, test_idx = next(splitter.split(X_feat, Y_orig, groups))

Xtr, Xte = X_feat[train_idx], X_feat[test_idx]
ytr, yte = Y_orig[train_idx], Y_orig[test_idx]

shared = set(groups[train_idx]) & set(groups[test_idx])
print('train %d / test %d, source photographs on both sides: %d (must be 0)'
      % (len(train_idx), len(test_idx), len(shared)))

clf = DecisionTreeClassifier(max_depth=3, class_weight="balanced",random_state=42).fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte),
                            target_names=["Healthy", "aculus", "peacock"]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(clf,
          feature_names=feature_names,
          class_names=["Healthy", "aculus_olearius", "olive_peacock_spot"],
          filled=True, rounded=True,
          impurity=False,        # hide gini
          proportion=True,       # percentages, clearer with class_weight="balanced"
          fontsize=10, ax=ax)
plt.tight_layout()
plt.savefig("olive_tree.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Saving the model
import joblib
joblib.dump(clf, 'leaf_tree.joblib')